# Phase 3 retained-pilot diagnostics

**Evidence status:** `pilot_adaptation` / `not_assessed`. This notebook reads retained ledgers and result records only. It neither runs a decoder nor makes a reproduction, ranking, superiority, or equal-total-cost claim.

In [1]:
import csv
import hashlib
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(os.environ['PHASE3_REPOSITORY_ROOT']).resolve()
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'bin' / 'python').resolve()
EVIDENCE_ROOT = ROOT / 'data' / 'results' / 'phase3-pilots'
REPORT_ROOT = ROOT / 'reports' / 'phase3-pilots'
RESULT_PATH = EVIDENCE_ROOT / 'result.json'
EVENTS_PATH = EVIDENCE_ROOT / 'events.jsonl'
for required in (RESULT_PATH, EVENTS_PATH):
    assert required.is_file(), f'missing retained evidence: {required}'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
LABEL = 'pilot_adaptation / not_assessed — diagnostic only; no reproduction, ranking, or equal-total-cost claim'


In [2]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
assert result['relation'] == 'pilot_adaptation'
assert result['assessment_status'] == 'not_assessed'
assert 'not statistical curve reproductions' in result['statistical_claim']
terminals = result['verified_terminals']
events = [json.loads(line) for line in EVENTS_PATH.read_text(encoding='utf-8').splitlines() if line]
assert terminals and events
assert {event['kind'] for event in events} >= {'shot_begin', 'work', 'runtime', 'termination'}
assert all(item['relation'] == 'pilot_adaptation' and item['assessment_status'] == 'not_assessed' for item in terminals)
assert all(len(item['artifact_raw_sha256']) == 64 and len(item['artifact_semantic_sha256']) == 64 and len(item['composition_identity']) == 64 for item in terminals)
evidence_digest = hashlib.sha256(RESULT_PATH.read_bytes() + EVENTS_PATH.read_bytes()).hexdigest()
assert all(item['budget_views']['equal_total_cost_claim'] is False for item in terminals)
len(terminals), Counter(item['termination'] for item in terminals)


(19, Counter({'max_iterations': 19}))

In [3]:
def policy_binding(item, role):
    return item['policy_effects'][role]['binding_id']

def trace_link(item):
    return f'data/results/phase3-pilots/events.jsonl#run_id={item["run_id"]};shot_id={item["shot_id"]}'

outcome_columns = [
    'run_id', 'code_id', 'update_policy', 'schedule_policy', 'termination', 'converged',
    'outcome', 'outcome_missing_reason', 'qualified_iterations', 'common_c2v_cap',
    'schedule_qualified_iteration_limit', 'equal_total_cost_claim', 'setup_runtime_ns',
    'decode_runtime_ns', 'denominator_runs', 'trace_evidence_link', 'evidence_sha256', 'relation', 'assessment_status',
]
work_columns = [
    'run_id', 'code_id', 'update_policy', 'schedule_policy', 'qualified_iterations',
    'full_work_vector_json', 'full_work_units', 'check_to_variable_updates',
    'variable_to_check_updates', 'convergence_checks', 'residual_priority_evaluations',
    'setup_runtime_ns', 'decode_runtime_ns', 'common_c2v_cap',
    'schedule_qualified_iteration_limit', 'equal_total_cost_claim', 'trace_evidence_link',
    'evidence_sha256', 'relation', 'assessment_status',
]
outcome_rows, work_rows = [], []
for item in terminals:
    work = item['work_vector']
    outcome = item['outcome']
    base = {
        'run_id': item['run_id'], 'code_id': item['code_id'],
        'update_policy': policy_binding(item, 'update'),
        'schedule_policy': policy_binding(item, 'schedule'),
        'qualified_iterations': json.dumps(item['qualified_iterations'], sort_keys=True),
        'common_c2v_cap': item['budget_views']['common_c2v_cap'],
        'schedule_qualified_iteration_limit': item['budget_views']['qualified_iteration_limit'],
        'equal_total_cost_claim': item['budget_views']['equal_total_cost_claim'],
        'setup_runtime_ns': item['setup_runtime']['value'],
        'decode_runtime_ns': item['decode_runtime']['value'],
        'trace_evidence_link': trace_link(item), 'evidence_sha256': evidence_digest,
        'relation': item['relation'], 'assessment_status': item['assessment_status'],
    }
    outcome_rows.append({**base, 'termination': item['termination'], 'converged': item['converged'],
        'outcome': outcome['class'] if outcome else 'missing',
        'outcome_missing_reason': 'unavailable_due_to_stopping' if outcome is None else '',
        'denominator_runs': len(terminals)})
    work_rows.append({**base, 'full_work_vector_json': json.dumps(work, sort_keys=True),
        'full_work_units': sum(value for value in work.values() if isinstance(value, (int, float))),
        'check_to_variable_updates': work.get('check_to_variable_updates', 0),
        'variable_to_check_updates': work.get('variable_to_check_updates', 0),
        'convergence_checks': work.get('convergence_checks', 0),
        'residual_priority_evaluations': work.get('residual_priority_evaluations', 0)})

for path, columns, rows in ((REPORT_ROOT / 'outcomes.csv', outcome_columns, outcome_rows), (REPORT_ROOT / 'work-by-policy.csv', work_columns, work_rows)):
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)
assert all((REPORT_ROOT / name).stat().st_size > 0 for name in ('outcomes.csv', 'work-by-policy.csv'))


In [4]:
labels = [item['run_id'].replace('phase3-', '') for item in terminals]
converged = [int(item['converged']) for item in terminals]
fig, axis = plt.subplots(figsize=(12, 4))
axis.bar(range(len(labels)), converged, color='#4477aa')
axis.set_xticks(range(len(labels)), labels, rotation=70, ha='right', fontsize=7)
axis.set_ylim(0, 1.1)
axis.set_ylabel('converged (retained pilot observation)')
axis.set_title('Phase 3 convergence diagnostics — ' + LABEL)
fig.tight_layout()
fig.savefig(REPORT_ROOT / 'convergence.png', dpi=160, metadata={'Title': 'Phase 3 ' + LABEL, 'Description': 'Retained evidence diagnostic; not a ranking or reproduction assessment.'})
plt.close(fig)

work_units = [row['full_work_units'] for row in work_rows]
setup_ms = [row['setup_runtime_ns'] / 1_000_000 for row in work_rows]
decode_ms = [row['decode_runtime_ns'] / 1_000_000 for row in work_rows]
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].bar(range(len(labels)), work_units, color='#66aa55')
axes[0].set_ylabel('full atomic work units')
axes[0].set_title('Phase 3 work and runtime diagnostics — ' + LABEL)
axes[1].bar(range(len(labels)), setup_ms, label='setup runtime', color='#cc6677')
axes[1].bar(range(len(labels)), decode_ms, bottom=setup_ms, label='decode runtime', color='#ddcc77')
axes[1].set_ylabel('runtime (ms), scopes stacked only for display')
axes[1].set_xticks(range(len(labels)), labels, rotation=70, ha='right', fontsize=7)
axes[1].legend()
fig.tight_layout()
fig.savefig(REPORT_ROOT / 'work-and-runtime.png', dpi=160, metadata={'Title': 'Phase 3 ' + LABEL, 'Description': 'Setup and decode scopes are separate; no equal-total-cost claim.'})
plt.close(fig)
assert all((REPORT_ROOT / name).stat().st_size > 0 for name in ('convergence.png', 'work-and-runtime.png'))


## Interpretation boundary

The tables and plots preserve missing outcomes, schedule-qualified iterations, full atomic work, and separate setup/decode timing scopes. They are bounded `pilot_adaptation` evidence with `not_assessed` status. Equal message caps or iteration limits do **not** imply equal total cost; no result here is a reproduction assessment or a decoder ranking.